# Experiment #9: Random Forest to Predict Titanic Survival

**Dataset:** Titanic passenger dataset (`titanic.csv`) - 891 passengers  
**Name:** Adithya Rajesh  |  **Class:** R3AI  |  **Roll No:** B25CTA07  |  **Date:** 13-10-2026

**Objective:** Build a **Random Forest classification model** to predict whether a passenger survived the Titanic disaster. We preprocess the data, select suitable features, train a forest of **100 decision trees**, evaluate it with accuracy and a classification report, and finally predict the survival status of a test passenger.

> **Note:** The notebook must stay in the same folder as `titanic.csv`.


## 1. Load the Dataset and Examine the Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

In [2]:
df = pd.read_csv('titanic.csv')
print('Shape:', df.shape)
df.head()

Shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
print('Data types:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isna().sum()[df.isna().sum() > 0])
print('\nSurvived distribution:')
print(df['Survived'].value_counts())

Data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

Missing values per column:
Age         177
Cabin       687
Embarked      2
dtype: int64

Survived distribution:
Survived
0    549
1    342
Name: count, dtype: int64


## 2. Data Preprocessing

- `PassengerId`, `Name` and `Ticket` are unique identifiers -> **dropped**.
- `Cabin` is missing for 687 of 891 passengers -> **dropped** (too incomplete to be useful).
- `Age` has 177 missing values -> filled with the **median** age.
- `Embarked` has 2 missing values -> filled with the **mode** (most common port).
- `Sex` and `Embarked` are categorical -> **encoded** as numbers.


In [4]:
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

print('Missing values remaining:', df.isna().sum().sum())
print('Shape:', df.shape)

Missing values remaining: 0
Shape: (891, 8)


## 3. Select the Features for Prediction

`Survived` is the target `y`; the predictors `X` are `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, `Fare` and `Embarked`.


In [5]:
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X = df[features]
y = df['Survived']

print('Features:', features)
print('Feature matrix shape:', X.shape)

Features: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
Feature matrix shape: (891, 7)


## 4. Split into Training and Testing Sets

We use a 75% / 25% split with `random_state=16` for reproducibility.


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=16)

print('Training set:', X_train.shape)
print('Test set:    ', X_test.shape)

Training set: (668, 7)
Test set:     (223, 7)


## 5. Train the Random Forest Classifier (100 Decision Trees)

`RandomForestClassifier` builds 100 decision trees and combines their votes.


In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=16)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print('Model trained successfully | Test predictions:', len(y_pred))

Model trained successfully | Test predictions: 223


## 6. Evaluate the Model - Accuracy and Classification Report

In [8]:
print('Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=['Died', 'Survived']))

Accuracy: 0.7982

              precision    recall  f1-score   support

        Died       0.82      0.87      0.84       137
    Survived       0.77      0.69      0.72        86

    accuracy                           0.80       223
   macro avg       0.79      0.78      0.78       223
weighted avg       0.80      0.80      0.80       223



In [9]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print('Feature importance:')
print(importances.round(3))

Feature importance:
Sex         0.274
Fare        0.268
Age         0.239
Pclass      0.089
SibSp       0.052
Parch       0.041
Embarked    0.038
dtype: float64


## 7. Predict the Survival Status of a Passenger from the Test Set

We take the **first passenger in the test set**, inspect their attributes, predict their survival and also read the probability from the forest.


In [10]:
passenger = X_test.iloc[[0]]
actual = y_test.iloc[0]

print('Passenger attributes:', passenger.values[0])
print('  Pclass:', passenger['Pclass'].values[0],
      '| Sex:', 'female' if passenger['Sex'].values[0] == 1 else 'male',
      '| Age:', passenger['Age'].values[0],
      '| SibSp:', passenger['SibSp'].values[0],
      '| Parch:', passenger['Parch'].values[0],
      '| Fare:', passenger['Fare'].values[0],
      '| Embarked:', {0: 'S', 1: 'C', 2: 'Q'}[passenger['Embarked'].values[0]])

prediction = rf.predict(passenger)[0]
probability = rf.predict_proba(passenger)[0]

print()
print('Actual survival      :', actual, '(did not survive)' if actual == 0 else '(survived)')
print('Predicted survival   :', prediction, '(did not survive)' if prediction == 0 else '(survived)')
print(f'P(did not survive)  : {probability[0]:.4f}')
print(f'P(survived)         : {probability[1]:.4f}')

Passenger attributes: [ 2.  1. 33.  0.  2. 26.  0.]
  Pclass: 2 | Sex: female | Age: 33.0 | SibSp: 0 | Parch: 2 | Fare: 26.0 | Embarked: S

Actual survival      : 1 (survived)
Predicted survival   : 1 (survived)
P(did not survive)  : 0.0300
P(survived)         : 0.9700


---
## Inference & Analysis

- **Accuracy** on the test set was **0.7982** (~80%), meaning most passengers were classified correctly by the 100-tree forest.
- **Classification report:** the model is better at recognising non-survivors (precision 0.82 / recall 0.87) than survivors (precision 0.77 / recall 0.69), reflecting the slightly larger number of passengers who died.
- **Feature importance:** `Sex` (0.274), `Fare` (0.268) and `Age` (0.239) are the strongest predictors - matching the historical reality of "women and children first" and the fact that higher-cost (first-class) tickets improve survival odds.
- **Test passenger interpretation:** a 33-year-old female in 2nd class, travelling with 2 family members and a moderately priced ticket (Fare 26), was predicted to **survive** with probability **0.97** - a highly confident prediction driven mainly by the gender feature. This shows the model output can be read both as a class and as a risk probability for each passenger.

---
**End of Experiment #9.**
